# Stress Prediction V28 — Near-Full Structure Search
V27의 Near-Full 검증기를 유지하고 ExtraTrees 구조 및 피처 선택 확률을 탐색합니다.

개발 4분할에서 25개 구조를 선별하고 상위 후보만 8개 감사 분할에서 1,200개 트리로 재검증합니다.

In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [2]:
from sklearn.model_selection import ShuffleSplit

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)
X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

PAIR_NAMES = list(combinations(PAIR_COLUMNS, 2))
DEV_TREES = 350
AUDIT_TREES = 1200
DEV_SPLITS = 4
AUDIT_SPLITS = 8
VALID_SIZE = 150
TREE_Q_GRID = (0.50, 0.52, 0.54)

COPY_SCHEMES = {
    "base": FEATURE_COPY_COUNTS,
    "lean": {k: max(0, v // 2) for k, v in FEATURE_COPY_COUNTS.items()},
    "none": {k: 0 for k in FEATURE_COPY_COUNTS},
    "work_focus": {
        "mean_working": 4, "bmi": 4, "cholesterol": 1, "height": 1,
        "glucose": 3, "weight": 1, "cholesterol_glucose_ratio": 1,
        "bone_density": 4,
    },
    "balanced": {k: 2 for k in FEATURE_COPY_COUNTS},
}
MAX_FEATURES_GRID = (1, 2, 3, "sqrt", 0.10)
STRUCTURES = [
    (scheme, max_features)
    for scheme in COPY_SCHEMES
    for max_features in MAX_FEATURES_GRID
]


def add_copies_by_scheme(frame, scheme_name):
    result = frame.copy()
    for feature, copy_count in COPY_SCHEMES[scheme_name].items():
        for copy_index in range(1, copy_count + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def pair_prediction(train_features, valid_features, target):
    train_rank, valid_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], valid_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(float),
            valid_features[column].to_numpy(float),
        )
    target_values = target.to_numpy(float)
    output = []
    for first, second in PAIR_NAMES:
        distance = (
            np.abs(valid_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(valid_rank[second][:, None] - train_rank[second][None, :])
        )
        output.append(target_values[np.argmin(distance, axis=1)])
    return np.quantile(np.column_stack(output), 0.52, axis=1)


def tree_quantiles(train_features, valid_features, target, scheme_name,
                   max_features, n_estimators, seed):
    weighted_train = add_copies_by_scheme(train_features, scheme_name)
    weighted_valid = add_copies_by_scheme(valid_features, scheme_name)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    valid_matrix = preprocessor.transform(weighted_valid)
    forest = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=max_features,
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(train_matrix, target)
    matrix = np.column_stack(
        [tree.predict(valid_matrix) for tree in forest.estimators_]
    )
    return {q: np.quantile(matrix, q, axis=1) for q in TREE_Q_GRID}


def blend(tree_prediction, pair_prediction_value):
    return np.clip(
        np.round(0.85 * tree_prediction + 0.15 * pair_prediction_value, 2),
        0.0, 1.0,
    )


splitter = list(ShuffleSplit(
    n_splits=DEV_SPLITS + AUDIT_SPLITS,
    test_size=VALID_SIZE,
    random_state=20260803,
).split(X))

# 1단계: 개발 분할에서 25개 구조를 350개 트리로 선별
dev_rows = []
for split_id, (train_idx, valid_idx) in enumerate(splitter[:DEV_SPLITS], 1):
    fold_train, fold_valid = X.iloc[train_idx], X.iloc[valid_idx]
    fold_target, valid_target = y.iloc[train_idx], y.iloc[valid_idx]
    pair_pred = pair_prediction(fold_train, fold_valid, fold_target)
    for scheme_name, max_features in STRUCTURES:
        q_predictions = tree_quantiles(
            fold_train, fold_valid, fold_target,
            scheme_name, max_features, DEV_TREES, 4200 + split_id,
        )
        for tree_q, tree_pred in q_predictions.items():
            prediction = blend(tree_pred, pair_pred)
            dev_rows.append({
                "split_id": split_id,
                "scheme": scheme_name,
                "max_features": max_features,
                "tree_q": tree_q,
                "MAE": mean_absolute_error(valid_target, prediction),
            })
    print(f"Development split {split_id}/{DEV_SPLITS} complete")

dev_frame = pd.DataFrame(dev_rows)
dev_summary = dev_frame.groupby(
    ["scheme", "max_features", "tree_q"], as_index=False, dropna=False
).MAE.mean().sort_values("MAE").reset_index(drop=True)

# 자료형이 섞인 max_features를 안전하게 고정하기 위해 dict 레코드로 보관
frozen = dev_summary.head(4).to_dict("records")
baseline_candidate = {"scheme": "base", "max_features": 1, "tree_q": 0.52}
if not any(
    row["scheme"] == "base"
    and str(row["max_features"]) == "1"
    and np.isclose(row["tree_q"], 0.52)
    for row in frozen
):
    frozen.append(baseline_candidate)

display(dev_summary.head(15))

# 2단계: 고정 후보만 8개 감사 분할, 트리 1,200개로 재검증
audit_rows = []
for audit_number, (train_idx, valid_idx) in enumerate(
    splitter[DEV_SPLITS:], 1
):
    fold_train, fold_valid = X.iloc[train_idx], X.iloc[valid_idx]
    fold_target, valid_target = y.iloc[train_idx], y.iloc[valid_idx]
    pair_pred = pair_prediction(fold_train, fold_valid, fold_target)
    prediction_cache = {}
    for row in frozen:
        scheme_name = row["scheme"]
        max_features = row["max_features"]
        key = (scheme_name, str(max_features))
        if key not in prediction_cache:
            prediction_cache[key] = tree_quantiles(
                fold_train, fold_valid, fold_target,
                scheme_name, max_features, AUDIT_TREES,
                7700 + audit_number,
            )
        tree_q = float(row["tree_q"])
        prediction = blend(prediction_cache[key][tree_q], pair_pred)
        audit_rows.append({
            "audit_split": audit_number,
            "scheme": scheme_name,
            "max_features": max_features,
            "tree_q": tree_q,
            "MAE": mean_absolute_error(valid_target, prediction),
        })
    print(f"Audit split {audit_number}/{AUDIT_SPLITS} complete")

audit_frame = pd.DataFrame(audit_rows)
baseline_scores = audit_frame[
    (audit_frame.scheme == "base")
    & (audit_frame.max_features.astype(str) == "1")
    & np.isclose(audit_frame.tree_q, 0.52)
].set_index("audit_split").MAE

summary_rows = []
for keys, part in audit_frame.groupby(
    ["scheme", "max_features", "tree_q"], dropna=False
):
    scheme_name, max_features, tree_q = keys
    scores = part.set_index("audit_split").MAE.sort_index()
    aligned_base = baseline_scores.loc[scores.index]
    improvements = aligned_base.to_numpy() - scores.to_numpy()
    summary_rows.append({
        "scheme": scheme_name,
        "max_features": max_features,
        "tree_q": tree_q,
        "audit_MAE": scores.mean(),
        "audit_mean_improvement": improvements.mean(),
        "audit_worst_improvement": improvements.min(),
        "audit_wins": int((improvements > 0).sum()),
    })

audit_summary = pd.DataFrame(summary_rows).sort_values(
    ["audit_wins", "audit_worst_improvement", "audit_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)
print("Baseline audit MAE:", f"{baseline_scores.mean():.6f}")
display(audit_summary)

eligible = audit_summary[
    ~(
        (audit_summary.scheme == "base")
        & (audit_summary.max_features.astype(str) == "1")
        & np.isclose(audit_summary.tree_q, 0.52)
    )
    & (audit_summary.audit_wins >= 6)
    & (audit_summary.audit_mean_improvement >= 0.0003)
    & (audit_summary.audit_worst_improvement >= -0.0005)
]

if eligible.empty:
    print("V28 rejected: 구조 후보가 Near-Full 감사 기준을 통과하지 못했습니다.")
    print("제출 파일을 생성하지 않습니다.")
    v28_accepted = False
else:
    selected = eligible.iloc[0]
    v28_accepted = True
    print("V28 accepted")
    print(selected)
    final_pair = pair_prediction(X, X_test, y)
    final_tree = tree_quantiles(
        X, X_test, y,
        selected["scheme"], selected["max_features"],
        AUDIT_TREES, 42,
    )[float(selected["tree_q"])]
    final_prediction = blend(final_tree, final_pair)
    final_submission = submission.copy()
    final_submission[TARGET] = final_prediction
    final_path = OUTPUT_DIR / "submit_v28_near_full_structure.csv"
    final_submission.to_csv(final_path, index=False, encoding="utf-8")
    print("Saved:", final_path)
    display(final_submission.head(10))



Development split 1/4 complete
Development split 2/4 complete
Development split 3/4 complete
Development split 4/4 complete


,scheme,max_features,tree_q,MAE
0,none,sqrt,0.50,0.119067
1,none,0.1,0.50,0.119117
2,none,3,0.50,0.119117
3,none,sqrt,0.52,0.119350
4,none,3,0.52,0.119717
5,none,0.1,0.52,0.119717
6,none,2,0.50,0.119850
7,none,sqrt,0.54,0.120167
8,balanced,2,0.52,0.120250
9,none,2,0.52,0.120367


Audit split 1/8 complete
Audit split 2/8 complete
Audit split 3/8 complete
Audit split 4/8 complete
Audit split 5/8 complete
Audit split 6/8 complete
Audit split 7/8 complete
Audit split 8/8 complete
Baseline audit MAE: 0.127633


,scheme,max_features,tree_q,audit_MAE,audit_mean_improvement,audit_worst_improvement,audit_wins
0,none,0.1,0.50,0.129800,-0.002167,-0.004400,1
1,none,3,0.50,0.129800,-0.002167,-0.004400,1
2,none,sqrt,0.52,0.130108,-0.002475,-0.004533,1
3,none,sqrt,0.50,0.130683,-0.003050,-0.007133,1
4,base,1,0.52,0.127633,0.000000,0.000000,0


V28 rejected: 구조 후보가 Near-Full 감사 기준을 통과하지 못했습니다.
제출 파일을 생성하지 않습니다.
